In [1]:
import duckdb
import pandas as pd
import csv
duckdb.sql("""
           CREATE OR REPLACE TABLE filtered AS
           SELECT message_id, channel_id, user_id, date, text_content, reply_to, n_forwards,
           views, reactions, forward_from, is_vaccine_related, language
           FROM read_json_auto('telegram.jsonl')
           WHERE text_content NOT NULL AND text_content != ''
           AND (language = 'Portuguese' OR language = 'English' OR language = 'Spanish')
           """)

# transform timestamp utc from mili to sec
duckdb.sql("UPDATE filtered SET date = date/1000")

# Limit the forward_from references to just the channels and users that had
# their messages tracked on the original dataset.
duckdb.sql("""
           UPDATE filtered
           SET forward_from = NULL
           WHERE forward_from NOT IN (
                SELECT DISTINCT channel_id
                FROM filtered
                WHERE channel_id IS NOT NULL)
           AND forward_from NOT IN (
                SELECT DISTINCT user_id
                FROM filtered
                WHERE user_id IS NOT NULL)
           """)
duckdb.sql("""
           UPDATE filtered
           SET n_forwards = 0.0
           WHERE forward_from = NULL
           OR reply_to = NULL
           """)

In [2]:
duckdb.sql("""
           SELECT language, COUNT(*) FROM filtered
           GROUP BY language
           """).show()

┌────────────┬──────────────┐
│  language  │ count_star() │
│  varchar   │    int64     │
├────────────┼──────────────┤
│ Spanish    │        71014 │
│ English    │       320762 │
│ Portuguese │      2331403 │
└────────────┴──────────────┘



# Channel table

In [3]:
# the reply_to field only contains references to messages within the same channel.
# because of that, only the forward_from field was considered. 
duckdb.sql("""
           CREATE OR REPLACE VIEW channels AS
           SELECT channel_id, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from NOT LIKE '<USER%'
           AND channel_id != forward_from
           """)

duckdb.sql("SELECT COUNT(*) FROM channels")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       214780 │
└──────────────┘

In [4]:
# Besides, group in pairs the message channel and the original channel (channel_id and
# forward_from) to compose the future graph edges.

duckdb.sql("""
           CREATE OR REPLACE VIEW channels2 AS
           SELECT channel_id, forward_from, count(*) AS weight
           FROM channels
           WHERE forward_from IN (
                SELECT DISTINCT channel_id
                FROM channels)
           GROUP BY channel_id, forward_from
           """)
duckdb.sql("SELECT COUNT(*) FROM channels2")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1570 │
└──────────────┘

In [33]:
duckdb.sql("COPY channels2 TO 'channel_edges.csv' (HEADER, DELIMITER ',')")


In [34]:
unique_ch = duckdb.sql("""
                       SELECT channel_id as channel_id FROM channels2
                       UNION
                       SELECT forward_from as channel_id FROM channels2
                       """).df()
duckdb.sql("COPY unique_ch TO 'channel_nodes.csv' (HEADER, DELIMITER ',')")

# Chat table

## Linking forwarding messages
As the forward_from field only specifies the channel from which the message came, it is necessary to search for the orginal forwarded message and link them.

In [6]:
# fwd_messages
duckdb.sql("""
           CREATE OR REPLACE VIEW fwd_messages AS
           SELECT message_id, channel_id, text_content, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from != ''
           AND forward_from not like '<USER%'
           """)

# the original message will be identified as two messages with same
# text content and one of them referencing the channel of the other.
duckdb.sql("""
           CREATE OR REPLACE TABLE fwd_reply_table AS
           SELECT a.message_id AS message_id, b.message_id AS fwd_reply
           FROM fwd_messages a JOIN filtered b
           ON a.text_content = b.text_content
           AND a.forward_from = b.channel_id
           AND a.message_id != b.message_id
           """)

In [7]:
#duckdb.sql("""COPY(
#           SELECT message_id, count(*)
#           FROM fwd_reply_table
#           GROUP BY message_id
#           ORDER BY count(*) DESC)
#           TO 'temp.csv' (HEADER, DELIMITER ',')
#           """)

In [8]:
# after manual inspection, it was seen that all messages that had high
# duplicity rates and were above 93 matches had their content only about
# channel link sharing or channel rules. It was decided to remove those
# links from analysis.
duckdb.sql("""
           DELETE FROM fwd_reply_table
           WHERE message_id IN (
                SELECT message_id
                FROM fwd_reply_table
                GROUP BY message_id
                HAVING COUNT(*) > 92
           )
           """)

print("Forwarded messages:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_messages"))
print("Total pairs origin-message:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_reply_table"))
print("Unique pairs origin-message:")
print(duckdb.sql("SELECT COUNT(distinct message_id) FROM fwd_reply_table"))

Forwarded messages:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       225704 │
└──────────────┘

Total pairs origin-message:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       307156 │
└──────────────┘

Unique pairs origin-message:
┌────────────────────────────┐
│ count(DISTINCT message_id) │
│           int64            │
├────────────────────────────┤
│                     210911 │
└────────────────────────────┘



In [9]:
duckdb.sql("""
           INSERT INTO fwd_reply_table (message_id, fwd_reply)
           SELECT a.message_id, a.reply_to
           FROM filtered AS a
           WHERE a.reply_to IS NOT NULL
           AND a.reply_to != ''
           """)
print("Total forward/reply references:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_reply_table"))

Total forward/reply references:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1129921 │
└──────────────┘



## Grouping messages into chats

In [10]:
# the following value is maximum time between messages of the same
# channel to be considered part of a same chat. It was defined after
# testing the number of chats and the distribution of chat duration.

# Eg: 2.5min collapsed all into 1.5million chats with 1-3 messages...
# 15min resulted on 630,000 chats, 9min mean, but more than half were single-message
# 60min resulted on 75% having 5 or less messages, but with a few enormous chats
chat_time_period = 1800 #seconds, eqv to 30min

In [11]:
# data from DuckDB into a Pandas DataFrame
df = duckdb.sql("SELECT * FROM filtered").df()
fwd_reply_df = duckdb.sql("SELECT * FROM fwd_reply_table").df()

df = df.sort_values(['channel_id', 'date'])

# time difference between consecutive messages in the same channel
# .diff() gives the difference with the previous row
df['time_gap'] = df.groupby('channel_id')['date'].diff()


# a new chat starts if time_gap > time OR if it's the first message in the channel (NaN)
# also limit chats to 3 hours
df['is_gap_break'] = (df['time_gap'] > chat_time_period) | (df['time_gap'].isna())
df['temp_session'] = df.groupby('channel_id')['is_gap_break'].cumsum()
df['session_start'] = df.groupby(['channel_id', 'temp_session'])['date'].transform('min')
df['seconds_since_start'] = df['date'] - df['session_start']
df['duration_lap'] = (df['seconds_since_start'] // 10800).astype(int)
df['chat_id'] = df.groupby(['channel_id', 'temp_session', 'duration_lap']).ngroup()

msg_to_chat_map = df.set_index('message_id')['chat_id'].to_dict()
fwd_reply_df['chat_id'] = fwd_reply_df['message_id'].map(msg_to_chat_map)
fwd_reply_df['fwd_reply_chat'] = fwd_reply_df['fwd_reply'].map(msg_to_chat_map)

In [12]:
df['time_gap'].divide(60).describe()

count    2.723060e+06
mean     7.908880e+01
std      1.540676e+03
min      0.000000e+00
25%      6.000000e-01
50%      2.333333e+00
75%      1.271667e+01
max      6.504263e+05
Name: time_gap, dtype: float64

In [13]:
df['chat_id'].max()

np.int64(470006)

In [14]:
# drop null and self references
links = fwd_reply_df.dropna(subset=['chat_id', 'fwd_reply_chat'])
links = links[links['chat_id'] != links['fwd_reply_chat']]

chat_links = (
    links.groupby('chat_id')['fwd_reply_chat']
    .apply(lambda x: list(set(x)))
    .reset_index()
)

In [15]:
link_sizes = chat_links['fwd_reply_chat'].apply(len)

mean_size = link_sizes.mean()
max_size = link_sizes.max()
median_size = link_sizes.median()

print(f"Mean size:   {mean_size:.2f}")
print(f"Max size:    {max_size}")
print(f"Median size: {median_size}")

print("\nFull Distribution:")
print(link_sizes.describe())

Mean size:   2.77
Max size:    203
Median size: 1.0

Full Distribution:
count    106235.000000
mean          2.769473
std           5.385702
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         203.000000
Name: fwd_reply_chat, dtype: float64


In [16]:
# Aggregate message-level data into chat-level data
chat_df = df.groupby('chat_id').agg({
    'channel_id': 'first',
    'message_id': 'count',
    'text_content': lambda x: " ".join(x.astype(str)),
    'user_id': lambda x: list(x.unique()),
    'views': 'mean',
    'reactions': 'sum',
    'n_forwards': 'sum',
    'date': ['min', 'max']
}).reset_index()

chat_df.columns = [
    'chat_id', 'channel_id', 'message_count', 'text_content', 'participants', 
    'avg_views', 'total_reactions', 'total_forwards', 'start_time', 'end_time'
]

# Merge the links into the chat_df
chat_df = chat_df.merge(chat_links, on='chat_id', how='left')

# Ensure 'fwd_reply_chat' is an empty list instead of NaN
chat_df['fwd_reply_chat'] = chat_df['fwd_reply_chat'].apply(lambda d: d if isinstance(d, list) else [])

chat_df['duration_seconds'] = chat_df['end_time'] - chat_df['start_time']

In [17]:
print("duration in minutes (hard limited to 3h)")
print(chat_df['duration_seconds'].divide(60).describe().apply("{0:.2f}".format))
print()
print(chat_df['message_count'].describe().apply("{0:.2f}".format))

duration in minutes (hard limited to 3h)
count    470007.00
mean         19.54
std          37.35
min           0.00
25%           0.00
50%           0.00
75%          23.35
max         179.98
Name: duration_seconds, dtype: str

count    470007.00
mean          5.79
std          18.97
min           1.00
25%           1.00
50%           2.00
75%           4.00
max        1479.00
Name: message_count, dtype: str


In [18]:
outliers = chat_df[(chat_df['duration_seconds'] > 86400)]
print(f"Number of chats exceeding 24 hours: {len(outliers)}")
outliers = chat_df[(chat_df['duration_seconds'] < 86400) &( chat_df['duration_seconds'] > 21600)]
print(f"Number of chats between 24h and 6h {len(outliers)}")

Number of chats exceeding 24 hours: 0
Number of chats between 24h and 6h 0


In [19]:
# filter out the isolated chats
sources = set(chat_links['chat_id'])
targets = set(item for sublist in chat_links['fwd_reply_chat'] for item in sublist)
source_target_ids = sources.union(targets)

# filter the main dataframe
final_nodes = chat_df[chat_df['chat_id'].isin(source_target_ids)].copy()

# for chat links, the source must be valid AND we only keep targets that still exist
final_links = chat_links[chat_links['chat_id'].isin(source_target_ids)].copy()
final_links['fwd_reply_chat'] = final_links['fwd_reply_chat'].apply(
    lambda targets: [t for t in targets if t in source_target_ids]
)

# Drop any rows that now have an empty list of links after filtering
final_links = final_links[final_links['fwd_reply_chat'].map(len) > 0]

In [20]:
final_nodes.drop(columns=['fwd_reply_chat'])
edges = final_links.explode('fwd_reply_chat')
edges.columns = ['chat_id', 'fwd_reply_chat']

print(f"Final Nodes: {len(final_nodes):,}")
print(f"Final Edges: {len(edges):,}")
print(f"Avg Connections per Chat: {len(edges)/len(final_nodes):.2f}")

Final Nodes: 175,040
Final Edges: 294,215
Avg Connections per Chat: 1.68


In [21]:
print("duration in minutes (hard limited to 3h)")
print(final_nodes['duration_seconds'].divide(60).describe().apply("{0:.2f}".format))
print()
print(final_nodes['message_count'].describe().apply("{0:.2f}".format))

duration in minutes (hard limited to 3h)
count    175040.00
mean         34.87
std          49.64
min           0.00
25%           0.00
50%          13.17
75%          47.23
max         179.98
Name: duration_seconds, dtype: str

count    175040.00
mean         10.80
std          29.75
min           1.00
25%           1.00
50%           3.00
75%           9.00
max        1479.00
Name: message_count, dtype: str


In [22]:
# export
final_nodes['text_content'] = (
    final_nodes['text_content']
    .str.replace(r'\n', ' ', regex=True)
    .str.replace('"', "'")
)

final_nodes.drop(columns=['fwd_reply_chat', 'participants']).to_csv(
    'chat_nodes.csv', 
    index=False, 
    quoting=csv.QUOTE_NONNUMERIC,
    escapechar='\\',
    encoding='utf-8'
)

edges.to_csv('chat_edges.csv', index=False)

# User table

Users will have a pondered, non-directed edge if they joined the same chat, or if they forwarded messages from one another

In [23]:
from itertools import combinations
from collections import Counter

In [24]:
df = duckdb.sql("SELECT * FROM filtered").df()
fwd_reply_df = duckdb.sql("SELECT * FROM fwd_reply_df").df()
author_map = df.set_index('message_id')['user_id'].to_dict()

fwd_reply_df['author_sender'] = fwd_reply_df['message_id'].map(author_map)
fwd_reply_df['author_receiver'] = fwd_reply_df['fwd_reply'].map(author_map)

fwd_reply_df = fwd_reply_df.dropna(subset=['author_sender', 'author_receiver'])
fwd_reply_df = fwd_reply_df[fwd_reply_df['author_sender'] != fwd_reply_df['author_receiver']]


In [25]:
chat_participants_flat = duckdb.sql("""
                                    SELECT chat_id, unnest(participants) AS user_id
                                    FROM final_nodes
                                    """).df()

In [26]:
# same chat
co_participation_edges = duckdb.sql("""
    SELECT 
        a.user_id as user_a, 
        b.user_id as user_b, 
        count(*) as co_weight
    FROM chat_participants_flat a
    JOIN chat_participants_flat b ON a.chat_id = b.chat_id
    WHERE a.user_id < b.user_id
    GROUP BY user_a, user_b
""").df()

In [28]:
# forward/reply
direct_edges = duckdb.sql("""
    SELECT 
        case when author_sender < author_receiver then author_sender else author_receiver end as user_a,
        case when author_sender < author_receiver then author_receiver else author_sender end as user_b,
        count(*) as direct_weight
    FROM fwd_reply_df
    WHERE author_sender IS NOT NULL AND author_receiver IS NOT NULL
      AND author_sender != author_receiver
    GROUP BY user_a, user_b
""").df()

In [30]:
# union
user_edges = duckdb.sql("""
    SELECT 
        COALESCE(c.user_a, d.user_a) as user_a,
        COALESCE(c.user_b, d.user_b) as user_b,
        (COALESCE(c.co_weight, 0) + COALESCE(d.direct_weight, 0)) as weight
    FROM co_participation_edges c
    FULL OUTER JOIN direct_edges d 
        ON c.user_a = d.user_a AND c.user_b = d.user_b
""").df()
unique_users = duckdb.sql("""
    SELECT user_a as user_id FROM user_edges
    UNION
    SELECT user_b as user_id FROM user_edges
""").df()

In [31]:
user_edges.to_csv('user_edges.csv', index=False, encoding='utf-8')
unique_users.to_csv('user_nodes.csv', index=False, encoding='utf-8')

In [32]:
user_edges['weight'].describe()

count    1.055406e+07
mean     1.388468e+00
std      5.952740e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      6.567000e+03
Name: weight, dtype: float64